In [1]:
from utils_thijs import recording_onsets, extract_all_spike_times_from_phy, split_spikes_by_recording, extract_cluster_groups, save_obj, detect_onsets, load_data, run_minimal_sanity_check

raw_dir = params.data_dir / params.sid / 'raw'
phy_dir = params.data_dir / params.sid / 'processed' / 'sorted'
recording_names = [f.name for f in raw_dir.iterdir() if f.suffix == '.raw']

In [2]:
rec_onsets = recording_onsets(recording_names, path=raw_dir)
cluster_number, good_clusters = extract_cluster_groups(phy_dir)
print(f"{len(good_clusters)} good clusters ({len(cluster_number)} total)\n")

print("Extracting spike times from phy...")
all_spike_times = extract_all_spike_times_from_phy(phy_dir)

print("Splitting spikes per recording, per neuron...")
good_data = split_spikes_by_recording(all_spike_times, good_clusters, rec_onsets)

save_name = params.output_directory / f'{params.sid}_fullexp_neurons_data.pkl'

save_obj(good_data, save_name)
print(f"\nSaved: {save_name}")

Extracting Manually Curated 'Good' clusters
22 good clusters (188 total)

Extracting spike times from phy...


100%|██████████| 7558773/7558773 [00:02<00:00, 3008147.67it/s]


Splitting spikes per recording, per neuron...


100%|██████████| 22/22 [00:05<00:00,  4.19it/s]


Saved: C:\thijs\sono_data\2026-09-09 mouse c57 758 Mekano6 A\processed\standard_analysis_pipeline\2026-09-09 mouse c57 758 Mekano6 A_fullexp_neurons_data.pkl


In [3]:
# from utils import run_minimal_sanity_check

for rec_name in recording_names:
    if 'checkerboard' not in rec_name:
        continue
    print(f"\n----- Triggers {rec_name}) -----")

    input_file = raw_dir / rec_name
    trigger_out_file = params.output_directory / f"{params.sid}_{rec_name}_triggers.pkl"
    data_out_file = params.output_directory / f"{params.sid}_{rec_name}_triggers_data.pkl"


    # Visual stimulus -> triggers are on the visual channel (no holography here).
    data, t_tot = load_data(
        input_path=input_file,
        dtype=params.dtype,
        nb_channels=params.nb_channels,
        channel_id=params.dmd_channel,
        probe_size=None,
        voltage_resolution=params.data_voltage_resolution,
    )


    indices = detect_onsets(data, params.dmd_threshold)
    # indices_errors = run_minimal_sanity_check(indices, sampling_rate=params.fs, maximal_jitter=params.maximal_jitter,)

    save_obj(
        {
            "indices": indices,
            "duration": t_tot,
            "trigger_type": 'dmd',
            "indice_errors": None,
        },
        trigger_out_file,
    )
    save_obj(data, data_out_file)



----- Triggers rec_1_A_20260909_checkerboard.raw) -----


100%|██████████| 18268000/18268000 [00:20<00:00, 886395.22it/s] 


In [5]:
from utils import analyse_checkerboard as analysis



rec_name = [f for f in recording_names if 'checkerboard' in f]
assert len(rec_name) == 1
rec_name = rec_name[0]

SWN = False  # Shifting white noise
checkerboard_output_dir = params.output_directory / 'checkerboard'

checkerboard_spikes, triggers, nb_repeats, cells_id, checkerboard = (
    analysis.load_checkerboard_data(
        rec_name=rec_name,
        check_directory=checkerboard_output_dir,
        checkerboard_params=checkerboard_params,
    )
)




-------- Creating all paths ---------

- /!\ Experiment root folder NOT found:
    /media/idv-s8/SSD Storage/20260729_VideoLSTA_calib/
    Paths are still defined, but no folder was created and no recording listed.
    Set 'root' in params.py to your experiment folder before running an analysis.

- phy (.GUI): using your override   /!\ (this path does not exist!)
    \media\idv-s8\SSD Storage\20260729_VideoLSTA_calib\RAW_DATA\20260729_meas_00_SWN_30Hz\20260729_meas_00_SWN_30Hz.GUI


AttributeError: module 'params_thijs' has no attribute 'triggers_directory'